# Generation of prompts for generating Dataset

## 1. imports 

In [10]:
import pandas as pd

## 2. Load sample data

## 3. Article creation

### 3.1 articles_all_commodities_baseline_seed7


In [1]:
"""
Prompt template for generating short synthetic texts about the
INFLATION rate and the UNEMPLOYMENT rate, conditioned on:
  - the date of the text
  - 21 trading days of history ending on that date, for EACH sampled
    series (both are percent rates, from data_gen.ipynb)
  - which of the two series the text is about (1 or 2)
The prompt hands the model four inputs: (1) the last 21 trading days of
each series, (2) the number of words (and sentences), (3) the label
dovish / neutral / hawkish (derived from the series, below) and (4) whether
the date lies in an IN period (inside a window of data_gen.ipynb: break,
extra variance or extra correlation) or an OUT period (normal times). The
period sets the COLOURING of the text (unusual episode vs. routine), the
label sets its policy stance. (5) The date itself: real developments
around that date may serve as a template for the setting, as long as they
do not contradict the label and the period.

LENGTH -- matched to the labelled FOMC sentences in
LENGTH_REFERENCE (data/test_data/lab-manual-combine-test-944601.xlsx):
each text draws one random row of that file and takes over its number of
words and number of sentences. Reference statistics (476 rows):
    words per entry : mean 31.2, median 28, sd 17.6, 10-90 % 14-50,
                      min 2, max 181 (a speech header, clipped below)
    characters      : mean 201, median 182
    letters per word: mean 5.4
    sentences       : 97 % one sentence, 3 % two or three
    by label        : dovish 31.5 / hawkish 31.0 / neutral 31.1 words
So a generated text is, like the reference data, usually ONE sentence
of about 30 words -- no headline.

STANCE -- given by the time series (compute_stance), not chosen by the model:
  score = 1.5*(inflation - 2) + 1.5*d_inflation              (inflation in the subject)
        - 1.0*(unemployment - 5) - 2.0*d_unemployment        (unemployment in the subject)
  levels = mean of the last 21 trading days, d_* = change of that mean
  against 3 months (63 trading days) earlier; only the series the text is
  about enter the score. score <= low -> dovish (label 0),
  score >= high -> hawkish (1), otherwise neutral (2) -- the same coding
  as the lab-manual data. Thresholds = 27 % / 75 % quantiles of the score
  on the baseline series 1970-2025 -> 27 % dovish, 48 % neutral,
  25 % hawkish over all days (the shares of the test set).
  (Taylor-rule logic: inflation above the 2 % target or rising and a tight
  labor market call for tighter policy; the opposite for easier policy.)

ROLE: the model writes as a member of the FOMC -- a passage of a statement,
minutes, press conference, testimony or speech (FOMC_DOCUMENTS, chosen to
fit the date), like the sentences of the lab-manual data.

RULES for the model:
  - write with the given stance
  - never name the series directly ("inflation", "CPI", "jobless", ...);
    refer to them only indirectly
  - never state figures; the price data only sets the TONE
  - with probability p_ambiguous_theme one shared macro theme is added

Usage:
    stance = compute_stance(series_df, date, commodities)
    prompt = build_prompt(date, commodities, prices_by_commodity, stance, seed=obs_id)
"""

import re

import numpy as np
import pandas as pd

# ── Length reference: labelled FOMC sentences ─────────────────────
LENGTH_REFERENCE = "../../data/test_data/lab-manual-combine-test-944601.xlsx"
_SENT_SPLIT = re.compile(r'(?<=[.!?])\s+(?=[A-Z"])')


def _load_length_reference(path: str = LENGTH_REFERENCE):
    s = pd.read_excel(path)["sentence"].astype(str).str.strip()
    words = s.str.split().str.len().to_numpy()
    n_sentences = s.apply(lambda x: max(1, len([p for p in _SENT_SPLIT.split(x) if p.strip()]))).to_numpy()
    return words, n_sentences


REF_WORDS, REF_SENTENCES = _load_length_reference()
MIN_WORDS = 5                                        # very short stubs ("Price shocks.") are kept to >= 5 words
MAX_WORDS = int(np.percentile(REF_WORDS, 99))        # cuts the speech-header outlier
print(f"Length reference: {len(REF_WORDS)} sentences, mean {REF_WORDS.mean():.1f} words "
      f"(median {np.median(REF_WORDS):.0f}, sd {REF_WORDS.std(ddof=1):.1f}), "
      f"{(REF_SENTENCES == 1).mean():.0%} single sentences; sampling clipped to [{MIN_WORDS}, {MAX_WORDS}] words")


def sample_length(rng: np.random.Generator) -> tuple[int, int]:
    """(number of words, number of sentences) of one random reference sentence."""
    i = int(rng.integers(len(REF_WORDS)))
    return int(np.clip(REF_WORDS[i], MIN_WORDS, MAX_WORDS)), int(REF_SENTENCES[i])


# ── Stance rule (see docstring) ───────────────────────────────────
INFLATION_TARGET  = 2.0    # %
NATURAL_RATE      = 5.0    # % unemployment
LEVEL_WINDOW      = 21     # trading days averaged for the level
CHANGE_LAG        = 63     # trading days (~3 months) for the change
STANCE_THRESHOLDS = {      # (low, high), calibrated on the baseline series (data_gen.ipynb, seed 7)
    ("inflation",):                (0.572, 4.524),
    ("unemployment",):             (-1.976, 0.016),
    ("inflation", "unemployment"): (-0.258, 4.222),
}
STANCE_LABELS = {"dovish": 0, "hawkish": 1, "neutral": 2}   # lab-manual coding
STANCE_DEFINITIONS = {
    "hawkish": "it points towards TIGHTER monetary policy (higher interest rates): price "
               "pressures are too strong or building and/or the labor market is tight",
    "dovish":  "it points towards EASIER monetary policy (lower interest rates): price "
               "pressures are subdued or fading and/or there is substantial labor-market slack",
    "neutral": "it has no clear implication for monetary policy: it reports developments "
               "factually, or the signals are balanced",
}


def compute_stance(series_df: pd.DataFrame, date: str, commodities: list[str]) -> dict:
    """
    Stance of the text, derived from the series up to `date`.
    series_df: full daily series indexed by date with columns <series>_price
    (as loaded by _load_baseline).
    """
    idx = series_df.index.get_indexer([pd.Timestamp(date)])[0]
    if idx < 0:
        raise ValueError(f"Date {date} not in the series")
    comps = {}
    for c in sorted(commodities):
        x      = series_df[f"{c}_price"].to_numpy(dtype=float)
        level  = float(x[max(0, idx - LEVEL_WINDOW + 1): idx + 1].mean())
        j      = max(0, idx - CHANGE_LAG)
        before = float(x[max(0, j - LEVEL_WINDOW + 1): j + 1].mean())
        comps[c] = {"level": level, "change_3m": level - before}
    score = 0.0
    if "inflation" in comps:
        score += 1.5 * (comps["inflation"]["level"] - INFLATION_TARGET) + 1.5 * comps["inflation"]["change_3m"]
    if "unemployment" in comps:
        score -= 1.0 * (comps["unemployment"]["level"] - NATURAL_RATE) + 2.0 * comps["unemployment"]["change_3m"]
    low, high = STANCE_THRESHOLDS[tuple(sorted(commodities))]
    stance = "dovish" if score <= low else "hawkish" if score >= high else "neutral"
    return {"stance": stance, "label": STANCE_LABELS[stance], "score": round(score, 3),
            "components": {c: {k: round(v, 3) for k, v in d.items()} for c, d in comps.items()}}


# ── IN / OUT period: colouring of the text ────────────────────────
# regime = which data_gen dataset the row comes from
PERIOD_COLOURING = {
    "IN": {
        "struct":   "an UNUSUAL EPISODE: the indicator(s) have just shifted abruptly to a new level "
                    "and are only slowly settling back -- colour the text with a sense of a sudden "
                    "break or shock: surprise, urgency, a marked change from what came before",
        "var":      "an UNUSUAL EPISODE of turbulence: the indicator(s) are swinging far more than "
                    "usual -- colour the text with uncertainty and nervousness: erratic readings, "
                    "conflicting signals, observers struggling to read the trend",
        "rho":      "an UNUSUAL EPISODE in which the two indicators move unusually closely together "
                    "-- colour the text with a sense that price developments and the job market are "
                    "being driven by one common force and reinforce each other",
        "baseline": "a period that market observers treat as an UNUSUAL EPISODE -- colour the text "
                    "with heightened attention and a sense that something out of the ordinary is "
                    "under way",
    },
    "OUT": "NORMAL TIMES: no unusual episode -- colour the text as routine and unexcited: "
           "steady developments, business as usual, nothing dramatic",
}


def _period_text(in_period: bool, regime: str) -> str:
    if not in_period:
        return "OUT -- " + PERIOD_COLOURING["OUT"]
    return "IN -- " + PERIOD_COLOURING["IN"].get(regime, PERIOD_COLOURING["IN"]["baseline"])


def _date_text(article_date: str) -> str:
    d = pd.Timestamp(article_date)
    return f"{d.strftime('%B')} {d.year}, {d.strftime('%A')}"


# ── Series-specific metadata ──────────────────────────────────────
COMMODITY_META = {
    "inflation": {
        "unit":      "% y/y",
        "price_col": "price_inflation",
        "family":    "macro",
        "signals": [
            ("energy and fuel costs feeding through to shop prices",            1950, 9999),
            ("wage-price spiral concerns as unions press for pay increases",        1950, 9999),
            ("a central bank weighing a rate decision against rising price pressures", 1950, 9999),
            ("expectations among households and firms about future price growth",   1960, 9999),
            ("food and grocery prices climbing faster than household budgets",      1900, 9999),
            ("rent and housing costs lifting the shelter component of broad price measures", 1960, 9999),
            ("supply chain bottlenecks pushing up the cost of goods",               1990, 9999),
            ("government price and wage controls meant to hold down price growth",  1971, 1980),
            ("the aftermath of the oil embargo shock still working through prices", 1973, 1982),
            ("tight monetary policy aimed at bringing price growth back to target", 1979, 9999),
            ("a weaker currency raising the cost of imported goods",                1950, 9999),
            ("fiscal stimulus and strong demand outrunning supply",                 2008, 9999),
            ("a pandemic-era rebound in demand meeting constrained supply",         2020, 9999),
            ("a sharp drop in fuel costs pulling headline price growth lower",      1950, 9999),
            ("core price measures stripping out food and energy staying sticky",    1980, 9999),
            ("bond markets pricing in higher long-term price growth",               1980, 9999),
            ("policymakers debating whether price pressures are transitory or persistent", 1970, 9999),
            ("a monthly price report coming in hotter than analysts had expected",  1950, 9999),
            ("a monthly price report coming in softer than analysts had expected",  1950, 9999),
            ("a risk of outright falling prices after a deep downturn",             1990, 9999),
        ],
        "forbidden": [
            '"inflation"', '"inflationary"', '"disinflation"', '"deflation"',
            '"CPI"', '"consumer price index"', '"consumer prices"', '"PCE"',
            '"price index"', '"cost of living"',
        ],
        "indirect": [
            ('"the pace at which everyday prices are rising"',            1900, 9999),
            ('"the erosion of purchasing power"',                         1900, 9999),
            ('"the figure households feel at the checkout"',              1900, 9999),
            ('"the broad measure of how fast the cost of a typical shopping basket moves"', 1950, 9999),
            ('"the yardstick central bankers watch to set policy"',       1950, 9999),
            ('"the gauge of price pressures across the economy"',         1950, 9999),
            ('"the headline price-growth reading"',                       1950, 9999),
            ('"the indicator the central bank is mandated to keep stable"', 1950, 9999),
            ('"the macro indicator in question"',                         1900, 9999),
            ('"the economic gauge in the spotlight"',                     1900, 9999),
        ],
    },
    "unemployment": {
        "unit":      "% of labor force",
        "price_col": "price_unemployment",
        "family":    "macro",
        "signals": [
            ("layoff announcements from large manufacturers",                      1900, 9999),
            ("a rising number of people filing new claims for state benefits",     1950, 9999),
            ("employers pulling back on hiring amid weak order books",             1900, 9999),
            ("a monthly employment report showing weaker-than-expected job creation", 1950, 9999),
            ("a monthly employment report showing stronger-than-expected job creation", 1950, 9999),
            ("plant closures hitting industrial regions hard",                     1900, 9999),
            ("youth and long-term out-of-work numbers drawing political attention", 1960, 9999),
            ("a tight labor market with employers struggling to fill vacancies",   1950, 9999),
            ("workers dropping out of the labor force and discouraged job seekers", 1950, 9999),
            ("government job-creation and retraining programs",                    1930, 9999),
            ("a recession deepening and spreading to services",                    1900, 9999),
            ("a recovery in hiring following a downturn",                          1900, 9999),
            ("automation and offshoring displacing factory workers",               1970, 9999),
            ("wage growth cooling as slack in the labor market builds",            1950, 9999),
            ("extended benefit programs for people out of work",                   1950, 9999),
            ("a sharp spike in claims after pandemic-related shutdowns",           2020, 2022),
            ("the central bank citing employment goals in its rate outlook",       1978, 9999),
            ("regional disparities in job availability between cities and rural areas", 1950, 9999),
        ],
        "forbidden": [
            '"unemployment"', '"unemployed"', '"jobless"', '"joblessness"',
            '"jobless rate"', '"jobless claims"', '"unemployment rate"',
            '"nonfarm payrolls"', '"payrolls"', '"jobs report"',
        ],
        "indirect": [
            ('"the share of the workforce without a job"',                1900, 9999),
            ('"the number of people looking for work"',                   1900, 9999),
            ('"the labor-market slack measure"',                          1950, 9999),
            ('"the gauge of how many willing workers lack employment"',   1950, 9999),
            ('"the monthly employment-market reading"',                   1950, 9999),
            ('"the indicator tied to the central bank\'s employment goal"', 1978, 9999),
            ('"the macro indicator in question"',                         1900, 9999),
            ('"the economic gauge in the spotlight"',                     1900, 9999),
        ],
    },
}

_ALL_COMMODITIES = list(COMMODITY_META.keys())

# Shared macro themes that fit both series (used with probability p_ambiguous_theme)
SHARED_THEMES = [
    ("investors reassessing the outlook for monetary policy",           1950, 9999),
    ("markets waiting for the next batch of official statistics",       1950, 9999),
    ("a debate over whether the economy is heading for a soft landing", 1970, 9999),
    ("economists revising their forecasts for the coming quarters",     1950, 9999),
    ("political pressure on policymakers ahead of an election",         1900, 9999),
    ("a divergence between the official figures and how households feel", 1950, 9999),
    ("central bank officials signalling a data-dependent approach",     1990, 9999),
    ("the trade-off between price stability and employment goals",      1960, 9999),
    ("a slowdown in consumer spending",                                 1950, 9999),
]


# Themes with a clear policy direction; a text never gets a theme that points
# the OTHER way than its stance (neutral texts may get any theme).
_HAWKISH_THEMES = {
    "energy and fuel costs feeding through to shop prices",
    "wage-price spiral concerns as unions press for pay increases",
    "food and grocery prices climbing faster than household budgets",
    "rent and housing costs lifting the shelter component of broad price measures",
    "supply chain bottlenecks pushing up the cost of goods",
    "the aftermath of the oil embargo shock still working through prices",
    "a weaker currency raising the cost of imported goods",
    "fiscal stimulus and strong demand outrunning supply",
    "a pandemic-era rebound in demand meeting constrained supply",
    "core price measures stripping out food and energy staying sticky",
    "bond markets pricing in higher long-term price growth",
    "a monthly price report coming in hotter than analysts had expected",
    "a monthly employment report showing stronger-than-expected job creation",
    "a tight labor market with employers struggling to fill vacancies",
    "a recovery in hiring following a downturn",
}
_DOVISH_THEMES = {
    "a sharp drop in fuel costs pulling headline price growth lower",
    "a monthly price report coming in softer than analysts had expected",
    "a risk of outright falling prices after a deep downturn",
    "layoff announcements from large manufacturers",
    "a rising number of people filing new claims for state benefits",
    "employers pulling back on hiring amid weak order books",
    "a monthly employment report showing weaker-than-expected job creation",
    "plant closures hitting industrial regions hard",
    "youth and long-term out-of-work numbers drawing political attention",
    "workers dropping out of the labor force and discouraged job seekers",
    "a recession deepening and spreading to services",
    "automation and offshoring displacing factory workers",
    "wage growth cooling as slack in the labor market builds",
    "extended benefit programs for people out of work",
    "a sharp spike in claims after pandemic-related shutdowns",
    "a slowdown in consumer spending",
}
_OPPOSITE = {"hawkish": _DOVISH_THEMES, "dovish": _HAWKISH_THEMES, "neutral": set()}


def _era_filter(themes: list[tuple[str, int, int]], year: int) -> list[str]:
    """Keep only themes plausible for the given year."""
    return [t for (t, y_from, y_to) in themes if y_from <= year <= y_to]


# FOMC communication the sentence is taken from: (document, speaker, first year, last year).
# Matches the sources of the lab-manual data (statements, minutes, press conferences, speeches).
FOMC_DOCUMENTS = [
    ("the FOMC's post-meeting policy statement",               "the Committee",                                   1994, 9999),
    ("the minutes of an FOMC meeting",                         "the staff secretariat summarising the Committee's discussion", 1993, 9999),
    ("the Record of Policy Actions of the FOMC",               "the staff secretariat summarising the Committee's discussion", 1936, 1992),
    ("the Chair's press conference after an FOMC meeting",     "the Chair of the Federal Reserve",                 2012, 9999),  # first one April 2011
    ("a speech by a member of the Board of Governors",         "a Governor of the Federal Reserve Board",          1936, 9999),
    ("a speech by a Reserve Bank president",                   "a Reserve Bank president who sits on the FOMC",   1936, 9999),
    ("the Chair's semiannual testimony to Congress",           "the Chair of the Federal Reserve",                 1978, 9999),
]


def _pick_document(rng: np.random.Generator, year: int) -> tuple[str, str]:
    docs = [(d, sp) for d, sp, y0, y1 in FOMC_DOCUMENTS if y0 <= year <= y1]
    d, sp = docs[int(rng.integers(len(docs)))]
    return d, sp


def build_prompt(
    article_date: str,
    commodities: list[str],
    prices_by_commodity: dict,
    stance: dict,
    in_period: bool = False,
    regime: str = "baseline",
    p_ambiguous_theme: float = 0.35,
    seed: int | None = None,
) -> dict:
    """
    Build the prompt for one short synthetic text.

    Parameters
    ----------
    article_date        : str   -- date of the text, e.g. "1987-10-20"
    commodities         : list  -- 1-2 of "inflation", "unemployment"
    prices_by_commodity : dict  -- {series: {"prices_21d": [...], "current_price": float}}
    stance              : dict  -- output of compute_stance(...) for this date/subject
    in_period           : bool  -- True if the date is inside a window (column IN == 1)
    regime              : str   -- "baseline" | "struct" | "var" | "rho" (dataset type)
    p_ambiguous_theme   : float -- probability of adding one shared macro theme
    seed                : int   -- random seed (use obs_id) for reproducibility

    Returns
    -------
    dict with keys "system", "prompt", "n_words", "n_sentences", "themes", "outlet"
    (= the FOMC document type),
    "stance", "stance_label"
    """
    commodities = [c.strip().lower() for c in commodities]
    if not commodities:
        raise ValueError("At least one series is required.")
    for c in commodities:
        if c not in COMMODITY_META:
            raise ValueError(f"Unknown series '{c}'. Must be one of {_ALL_COMMODITIES}.")
        if c not in prices_by_commodity:
            raise ValueError(f"Missing price data for '{c}' in prices_by_commodity.")

    rng                  = np.random.default_rng(seed)
    n_words, n_sentences = sample_length(rng)
    document, speaker    = _pick_document(rng, int(article_date[:4]))
    outlet               = document
    article_year         = int(article_date[:4])

    # ── Narrative themes ─────────────────────────────────────────────
    chosen_themes = []
    for c in commodities:
        signals = [t for t in _era_filter(COMMODITY_META[c]["signals"], article_year)
                   if t not in _OPPOSITE[stance["stance"]]]
        if signals:
            n_signals = int(rng.integers(1, 3))
            chosen_themes += rng.choice(signals, size=min(n_signals, len(signals)), replace=False).tolist()
    if rng.random() < p_ambiguous_theme:
        shared = [t for t in _era_filter(SHARED_THEMES, article_year) if t not in _OPPOSITE[stance["stance"]]]
        if shared:
            chosen_themes.append(str(rng.choice(shared)))
    # a one-sentence text cannot carry many themes: keep at most one per sentence (+1)
    chosen_themes = chosen_themes[: n_sentences + 1]

    # ── Naming rules ─────────────────────────────────────────────────
    forbidden, indirect = [], []
    for c in commodities:
        forbidden += COMMODITY_META[c]["forbidden"]
        indirect  += _era_filter(COMMODITY_META[c]["indirect"], article_year)

    subjects_str = ", ".join(commodities)
    sentences_str = "ONE sentence" if n_sentences == 1 else f"{n_sentences} sentences"

    # ── Input 1: the last 21 trading days of each series ─────────────
    data_lines = []
    for c in commodities:
        hist = [round(v, 2) for v in prices_by_commodity[c]["prices_21d"]]
        data_lines.append(
            f"  {c.capitalize()} ({COMMODITY_META[c]['unit']}), last {len(hist)} trading days, "
            f"oldest -> {article_date}:\n    {hist}"
        )
    data_block = "\n".join(data_lines)
    themes_line = f"  Themes you may reflect: {', '.join(chosen_themes)}\n" if chosen_themes else ""

    system = (
        "You are a member of the Federal Open Market Committee (FOMC), the body of the "
        "Federal Reserve System that sets U.S. monetary policy: the Chair, the other members of "
        "the Board of Governors and the presidents of the Federal Reserve Banks. "
        f"You are writing a passage of {document}, in the voice of {speaker}, "
        "as of the given date -- you only know what has happened up to and including that date. "
        "Use the measured, precise register of official Federal Reserve communication: "
        "the Committee's assessment of economic conditions, the outlook and the risks, and what "
        "they imply for the stance of monetary policy (hawkish, dovish or neutral, as instructed). "
        "The passage is very short: usually a single sentence. "
        "You never name the economic indicators it is about; you refer to them only indirectly. "
        "You never state figures -- the data you are given only informs your assessment. "
        "You may refer to real institutions and draw on real economic and political developments "
        "of the time as a template, as long as they happened on or before the date and do not "
        "contradict the given label and period."
    )

    prompt = f"""Write one passage of {document} ({speaker}), dated {article_date}.

INPUT 1 -- DATA (for your understanding only; never state these figures):
  Subject (internal only): {subjects_str}
{data_block}
  Reference points: price-stability goal about 2%, normal level of the job-seeker share about 5%.
{themes_line}
INPUT 2 -- LENGTH: exactly {sentences_str}, about {n_words} words in total.

INPUT 3 -- LABEL: {stance["stance"].upper()} -- {STANCE_DEFINITIONS[stance["stance"]]}.
  The label is fixed; a reader must be able to tell it from the wording alone.

INPUT 4 -- PERIOD: {_period_text(in_period, regime)}.
  The period only sets the colouring; the policy stance is always the label above.

INPUT 5 -- DATE: {article_date} ({_date_text(article_date)}).
  Use real economic, political and policy developments around this date as a template
  for the setting (e.g. what the central bank, governments, energy markets or the
  business cycle were doing at the time). Nothing may come from after this date.
  If the real situation then would point to a different stance or mood than INPUTS 3
  and 4, the inputs win: keep the real backdrop only where it fits, otherwise stay general.

RULES:
- Never write any of these words or a direct synonym, ticker or abbreviation: {', '.join(forbidden)}.
  Refer to the subject only indirectly, e.g. {' or '.join(indirect[:2])}.
- No numbers, percentages or figures. Reflect the data only through qualitative language
  ("edged higher", "eased", "remained elevated"), consistent with the label.
- A well-informed reader must be able to tell from context that the text is about
  {subjects_str}{' (both of them)' if len(commodities) > 1 else ''}.
- Everything must be plausible for {article_date}: no later events, institutions, technologies or vocabulary.
- Output only the text itself: no headline, no quotation marks, no preamble.
"""

    return {
        "system":      system,
        "prompt":      prompt.strip(),
        "n_words":     n_words,
        "n_sentences": n_sentences,
        "themes":      chosen_themes,
        "outlet":      outlet,
        "stance":      stance["stance"],
        "stance_label": stance["label"],
        "in_period":   int(bool(in_period)),
        "regime":      regime,
    }

Length reference: 476 sentences, mean 31.2 words (median 28, sd 17.6), 97% single sentences; sampling clipped to [5, 79] words


In [ ]:
import asyncio
import json
import os
from pathlib import Path

import anthropic
import nest_asyncio
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from tqdm.asyncio import tqdm_asyncio

# Claude via the Anthropic API (key: ANTHROPIC_API_KEY in the repo's .env)
MODEL        = "claude-haiku-5-5"   # newest Haiku: $0.10 / $0.50 per million input / output tokens
EFFORT       = "medium"   # low | medium | high | xhigh | max  (thinking depth, the model's default is medium)
MAX_TOKENS   = 16000      # thinking counts towards max_tokens, so leave room above the article itself
MAX_PARALLEL = 10         # concurrent requests
MAX_RETRIES  = 5          # the SDK retries 429 / 5xx / connection errors with exponential backoff
# No temperature: Claude Haiku 5.5 rejects non-default sampling parameters (temperature/top_p/top_k).
# Variety between articles comes from the randomised prompt (themes, outlet, length).

# Price history comes from the FULL continuous series written by data_gen.ipynb
# (one row per business day, 1970-2025; columns <series>_price), e.g.
#   ../data/stock_prices/full_series/full_baseline_inflation_unemployment_trial_seed7.csv
#   ../data/stock_prices/full_series/full_articles_inflation_unemployment_{struct|var|rho}_{low|medium|high}_seed7.csv
# -> genuine 21-trading-day lookback (the 5000-row datasets are sparse and only
#    carry a price where the series is a topic).
PRICE_COLS = {c: f"{c}_price" for c in ("inflation", "unemployment")}
LOOKBACK_DAYS = 21


def _load_api_key():
    """Anthropic key from the repo's .env (searched upwards from the notebook folder)."""
    for d in (Path.cwd(), *Path.cwd().parents):
        if (d / ".env").exists():
            load_dotenv(dotenv_path=d / ".env")
            break
    key = os.getenv("ANTHROPIC_API_KEY")
    if not key:
        raise ValueError("ANTHROPIC_API_KEY not found -- add it to the .env in the repo root")
    return key


def _make_client():
    return anthropic.AsyncAnthropic(api_key=_load_api_key(), max_retries=MAX_RETRIES)


def _load_baseline(csv_path):
    """Load a FULL continuous series (date, inflation_price, unemployment_price, ...)."""
    df = pd.read_csv(csv_path, parse_dates=["date"])
    missing = [c for c in PRICE_COLS.values() if c not in df.columns]
    if missing:
        raise ValueError(f"{csv_path} is not a full series file (missing {missing}) -- "
                         "pass the matching file from ../data/stock_prices/full_series/")
    return df.sort_values("date").set_index("date")


def _get_prices(df, date, price_col):
    """Window of LOOKBACK_DAYS trading days ENDING on `date` (so a jump on the
    article date shows up in trend/range) and the value ON `date` (in %)."""
    idx = df.index.get_indexer([pd.Timestamp(date)])[0]
    if idx < 0:
        raise ValueError(f"Date {date} not found in the full series")
    series        = df[price_col].to_numpy(dtype=float)
    prices_21d    = series[max(0, idx - LOOKBACK_DAYS + 1): idx + 1].tolist()
    current_price = float(series[idx])
    return prices_21d, current_price


async def _call_api(client, prompt_result):
    """
    One Messages API call. Returns (text, stop_reason, model that answered).

    Claude Haiku 5.5 thinks adaptively by default (thinking blocks come first,
    so the text is read by block type). It has no server-side refusal
    fallback: a declined request comes back with stop_reason "refusal" and
    the text stays empty. output_config goes through extra_body because the
    pinned SDK (anthropic==0.58.2) does not know this field yet.
    """
    response = await client.messages.create(
        model      = MODEL,
        max_tokens = MAX_TOKENS,
        system     = prompt_result["system"],
        messages   = [{"role": "user", "content": prompt_result["prompt"]}],
        extra_body = {"output_config": {"effort": EFFORT}},
    )
    text = "".join(b.text for b in response.content if getattr(b, "type", None) == "text").strip()
    if response.stop_reason == "refusal":
        print(f"  refusal ({getattr(response, 'stop_details', None)}) -- article left empty")
    elif response.stop_reason == "max_tokens":
        print(f"  hit max_tokens={MAX_TOKENS} -- article may be cut off")
    return text, response.stop_reason, response.model


async def _process_row(client, row, baseline_df, semaphore, build_prompt, regime):
    async with semaphore:
        date_str = str(row["date"])[:10]
        obs_id   = int(row["obs_id"])

        # ── one-hot commodities, read directly from sample_df ──────────
        commodities_onehot = {c: int(row[c]) for c in ("inflation", "unemployment")}
        parts = [c for c, v in commodities_onehot.items() if v == 1]

        if not parts:
            raise ValueError(f"obs_id {obs_id}: no commodity flagged (all zero)")

        # ── price history for every sampled commodity ─────────────────
        prices_by_commodity = {}
        for c in parts:
            p21, cur = _get_prices(baseline_df, date_str, PRICE_COLS[c])
            prices_by_commodity[c] = {"prices_21d": p21, "current_price": cur}

        # ── stance (dovish / neutral / hawkish) given by the series ───
        stance = compute_stance(baseline_df, date_str, parts)

        prompt_result = build_prompt(
            article_date        = date_str,
            commodities         = parts,
            prices_by_commodity = prices_by_commodity,
            stance              = stance,
            in_period           = bool(row["IN"]),
            regime              = regime,
            seed                = obs_id,
        )

        raw, stop_reason, served_by = await _call_api(client, prompt_result)
        # short text without headline (like the labelled FOMC sentences);
        # "headline" stays empty so eval/convert_jsons_to_csvs.py keeps working
        body = " ".join(l.strip() for l in raw.split("\n") if l.strip())

        return {
            "metadata": {
                "obs_id":           obs_id,
                "article_date":     date_str,
                "commodities":      commodities_onehot,
                "commodity":        " and ".join(parts),
                "prices": {
                    c: {"current_price": round(d["current_price"], 4),
                        "prices_21d":    [round(p, 4) for p in d["prices_21d"]]}
                    for c, d in prices_by_commodity.items()
                },
                "stance":           stance["stance"],
                "label":            stance["label"],        # 0 dovish, 1 hawkish, 2 neutral (lab-manual coding)
                "stance_score":     stance["score"],
                "stance_inputs":    stance["components"],
                "IN":               int(row["IN"]),           # 1 = inside a window of data_gen.ipynb
                "regime":           regime,
                "n_words_target":   prompt_result["n_words"],
                "n_sentences_target": prompt_result["n_sentences"],
                "n_words_actual":   len(body.split()),
                "themes":           prompt_result["themes"],
                "outlet":           prompt_result["outlet"],
                "model":            served_by,
                "stop_reason":      stop_reason,
            },
            "label":    stance["stance"],               # "dovish" | "neutral" | "hawkish"
            "label_id": stance["label"],                # 0 dovish, 1 hawkish, 2 neutral (lab-manual coding)
            "headline": "",
            "body":     body,
        }


async def _generate_all_async(sample_df, baseline_csv, out_path,  # baseline_csv = FULL series file
                               build_prompt, regime):
    baseline_df = _load_baseline(baseline_csv)
    semaphore   = asyncio.Semaphore(MAX_PARALLEL)
    os.makedirs(os.path.dirname(out_path), exist_ok=True)

    async with _make_client() as client:
        tasks = [
            _process_row(client, row, baseline_df, semaphore, build_prompt, regime)
            for _, row in sample_df.iterrows()
        ]
        results = await tqdm_asyncio.gather(*tasks, desc="Generating articles")

    articles = sorted(results, key=lambda x: x["metadata"]["obs_id"])

    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(articles, f, indent=2, ensure_ascii=False)

    print(f"\nSaved {len(articles)} articles → {out_path}")
    return articles


def generate_all(sample_df, dataset_csv, baseline_csv, build_prompt,
                 out_path=None, n_samples=None, sample_seed=7, regime="baseline"):
    """regime: "baseline" | "struct" | "var" | "rho" -- sets the IN-period colouring in the prompt."""
    if n_samples is not None:
        if n_samples > len(sample_df):
            raise ValueError(f"n_samples={n_samples} exceeds available rows ({len(sample_df)})")
        sample_df = (sample_df.sample(n=n_samples, random_state=sample_seed)
                     .sort_values("obs_id").reset_index(drop=True))
    if out_path is None:
        from datetime import datetime
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename  = f"{Path(dataset_csv).stem}_n{len(sample_df)}_{timestamp}.json"
        out_path  = str(Path(dataset_csv).parent.parent / "articles" / "jsons" / filename)
    nest_asyncio.apply()
    return asyncio.run(_generate_all_async(
        sample_df, baseline_csv, out_path, build_prompt, regime))

In [ ]:
# Datasets written by data_gen.ipynb (run it first)
BASELINE_CSV    = "../data/stock_prices/articles_all_commodities_baseline_seed7.csv"            # 5000 article rows
BASELINE_SERIES = "../data/stock_prices/full_series/full_baseline_inflation_unemployment_trial_seed7.csv"  # every business day

# Set to an int to generate only a random subsample of this many rows
# instead of the whole dataset; set to None to use every row.
N_SAMPLES = 200

baseline_df = (pd.read_csv(BASELINE_CSV, parse_dates=["date"])
                 .sort_values("date").reset_index(drop=True))

missing = [c for c in ["price_inflation", "price_unemployment"]
           if c not in baseline_df.columns]
if missing:
    raise ValueError(f"Baseline CSV is missing price columns: {missing}")

# The whole dataset, no sampling/balancing at all -- use every row as-is.
sample_df = baseline_df[["obs_id", "date", "inflation", "unemployment", "IN"]].copy()
sample_df["date"] = sample_df["date"].dt.strftime("%Y-%m-%d")

n_flagged = sample_df[["inflation", "unemployment"]].sum(axis=1)
n_empty   = (n_flagged == 0).sum()
if n_empty:
    print(f"Warning: {n_empty} rows have no commodity flagged -- dropping them.")
    sample_df = sample_df[n_flagged > 0].reset_index(drop=True)

print(f"Total articles available: {len(sample_df)}")
print(sample_df[["inflation", "unemployment"]].sum())

combo_counts = sample_df.apply(
    lambda r: "+".join(c for c in ("inflation","unemployment") if r[c] == 1), axis=1
).value_counts()
print("\nCombination distribution:")
print(combo_counts)

articles = generate_all(
    sample_df    = sample_df,
    dataset_csv  = BASELINE_CSV,
    baseline_csv = BASELINE_SERIES,   # full daily series -> 21-day price history
    build_prompt = build_prompt,
    n_samples    = N_SAMPLES,
    regime       = "baseline",
)

### 3.2 All other datasets

In [ ]:
# Articles for the struct / var / rho datasets of data_gen.ipynb.
# Every dataset is paired with ITS OWN full series (breaks, extra variance
# or correlation are only in that series), so the 21-day price history in
# the prompt shows the effect.
RUN_OTHER_DATASETS = False   # True -> calls the API (N_SAMPLES rows per dataset, 9 datasets)
N_SAMPLES_OTHER    = 200

DATA_DIR = Path("../data/stock_prices")
OTHER_DATASETS = {
    f"{kind}_{level}": (DATA_DIR / f"articles_inflation_unemployment_{kind}_{level}_seed7.csv",
                        DATA_DIR / "full_series" / f"full_articles_inflation_unemployment_{kind}_{level}_seed7.csv")
    for kind in ("struct", "var", "rho") for level in ("low", "medium", "high")
}

missing = [str(p) for pair in OTHER_DATASETS.values() for p in pair if not p.exists()]
if missing:
    raise FileNotFoundError("run data_gen.ipynb first, missing:\n  " + "\n  ".join(missing))

articles_by_dataset = {}
for name, (dataset_csv, series_csv) in OTHER_DATASETS.items():
    df = pd.read_csv(dataset_csv, parse_dates=["date"]).sort_values("date").reset_index(drop=True)
    sample_df = df[["obs_id", "date", "inflation", "unemployment", "IN"]].copy()
    sample_df["date"] = sample_df["date"].dt.strftime("%Y-%m-%d")
    sample_df = sample_df[sample_df[["inflation", "unemployment"]].sum(axis=1) > 0].reset_index(drop=True)
    print(f"{name:14s}: {len(sample_df)} rows  (IN={int(df['IN'].sum())})  series={series_csv.name}")
    if RUN_OTHER_DATASETS:
        articles_by_dataset[name] = generate_all(
            sample_df    = sample_df,
            dataset_csv  = str(dataset_csv),
            baseline_csv = str(series_csv),
            build_prompt = build_prompt,
            n_samples    = N_SAMPLES_OTHER,
            regime       = name.split("_")[0],   # struct / var / rho
        )
if not RUN_OTHER_DATASETS:
    print("\nRUN_OTHER_DATASETS = False -> nothing generated (set it to True to call the API).")
